# DSSM 调试 Notebook

Deep Structured Semantic Model - 深度结构化语义模型

双塔结构：用户塔 + 物品塔，通过向量相似度召回

In [23]:
# 导入必要的库
import os
import sys
import pickle
import math
import multiprocessing as mp
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import faiss
from sklearn.preprocessing import LabelEncoder

# 添加项目根目录到路径
project_root = Path(os.getcwd()).parent
sys.path.insert(0, str(project_root))
print(f"项目根目录: {project_root}")

from util.utils import Logger, gen_user_item_time
from Recall.Recall_Methods import evaluate

print("✅ 基础库导入成功！")

项目根目录: /home/pyy/XWTJ-main
✅ 基础库导入成功！


In [24]:
# 检查 deepmatch 是否可用（DSSM模型依赖）
try:
    from deepmatch.models import DSSM
    from deepmatch.utils import sampledsoftmaxloss, NegativeSampler
    from deepctr.feature_column import SparseFeat, VarLenSparseFeat, DenseFeat
    from tensorflow.python.keras.models import Model
    print("✅ DeepMatch 和 TensorFlow 导入成功！")
except ImportError as e:
    print(f"⚠️ 缺少依赖: {e}")
    print("请安装: pip install deepmatch tensorflow")

✅ DeepMatch 和 TensorFlow 导入成功！


In [3]:
# ==================== 路径配置 ====================
DATA_PATH = Path('../Initial_data')
MODEL_PATH = Path('../Results/Model')
RECALL_DICT_PATH = Path('../Results/Recall_dict')
SAVE_DIR = Path('../Results/DSSM')

TRAIN_DATA_PATH = DATA_PATH / 'train_click_log.csv'
TEST_DATA_PATH = DATA_PATH / 'testA_click_log.csv'
ITEM_EMB_PATH = DATA_PATH / 'articles_emb.csv'
ITEM_INFO_PATH = DATA_PATH / 'articles.csv'

print("路径配置:")
for p in [TRAIN_DATA_PATH, TEST_DATA_PATH, ITEM_INFO_PATH]:
    print(f"  {p.name}: {p.exists()}")

路径配置:
  train_click_log.csv: True
  testA_click_log.csv: True
  articles.csv: True


## 步骤1：加载数据

In [26]:
# 加载数据
train_data = pd.read_csv(TRAIN_DATA_PATH)
test_data = pd.read_csv(TEST_DATA_PATH)
item_raw_emb = pd.read_csv(ITEM_EMB_PATH)
item_info = pd.read_csv(ITEM_INFO_PATH)

print(f"训练数据: {train_data.shape}")
print(f"测试数据: {test_data.shape}")
print(f"物品信息: {item_info.shape}")
print(f"物品Embedding: {item_raw_emb.shape}")

训练数据: (1112623, 9)
测试数据: (518010, 9)
物品信息: (364047, 4)
物品Embedding: (364047, 251)


In [27]:
# 查看数据结构
print("=== 训练数据 ===")
print(train_data.head())
print(f"\n列: {train_data.columns.tolist()}")

=== 训练数据 ===
   user_id  click_article_id  click_timestamp  click_environment  \
0   199999            160417    1507029570190                  4   
1   199999              5408    1507029571478                  4   
2   199999             50823    1507029601478                  4   
3   199998            157770    1507029532200                  4   
4   199998             96613    1507029671831                  4   

   click_deviceGroup  click_os  click_country  click_region  \
0                  1        17              1            13   
1                  1        17              1            13   
2                  1        17              1            13   
3                  1        17              1            25   
4                  1        17              1            25   

   click_referrer_type  
0                    1  
1                    1  
2                    1  
3                    5  
4                    5  

列: ['user_id', 'click_article_id', 'click_timest

In [6]:
print("=== 物品信息 ===")
print(item_info.head())
print(f"\n列: {item_info.columns.tolist()}")

=== 物品信息 ===
   article_id  category_id  created_at_ts  words_count
0           0            0  1513144419000          168
1           1            1  1405341936000          189
2           2            1  1408667706000          250
3           3            1  1408468313000          230
4           4            1  1407071171000          162

列: ['article_id', 'category_id', 'created_at_ts', 'words_count']


In [7]:
# 数据预处理
item_info = item_info.rename(columns={'article_id': 'click_article_id'})

# Min-Max归一化
item_info['words_count'] = (item_info['words_count'] - item_info['words_count'].min()) / (item_info['words_count'].max() - item_info['words_count'].min())
item_info['created_at_ts'] = (item_info['created_at_ts'] - item_info['created_at_ts'].min()) / (item_info['created_at_ts'].max() - item_info['created_at_ts'].min())

# 合并全量数据
all_data = pd.concat([train_data, test_data], axis=0)
all_data = all_data.merge(item_info, how='left', on='click_article_id')
all_data = all_data.sort_values(by=['user_id', 'click_timestamp']).reset_index(drop=True)

print(f"合并后数据: {all_data.shape}")
print(f"列: {all_data.columns.tolist()}")

合并后数据: (1630633, 12)
列: ['user_id', 'click_article_id', 'click_timestamp', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type', 'category_id', 'created_at_ts', 'words_count']


In [8]:
# 获取热门物品
hot_item = all_data['click_article_id'].value_counts().index[:350]
print(f"热门物品数: {len(hot_item)}")
print(f"Top 10: {hot_item[:10].tolist()}")

热门物品数: 350
Top 10: [272143, 234698, 123909, 336221, 96210, 336223, 183176, 168623, 162655, 331116]


## 步骤2：特征工程

In [9]:
# 提取用户特征（环境、设备等众数）
features = ['click_environment', 'click_deviceGroup', 'click_os',
            'click_country', 'click_region', 'click_referrer_type']

#对每个特征，计算每个用户的 众数 （最频繁出现的值）。
# - 一个用户可能有多次点击，每次环境可能不同
# - 取 最常见的环境 作为该用户的特征
for feature in features:
    mode_data = all_data.groupby('user_id')[feature].apply(lambda x: x.mode()[0]).reset_index()
    mode_data.rename(columns={feature: f'{feature}_mode'}, inplace=True)
    all_data = pd.merge(all_data, mode_data, on='user_id', how='left')

print("✅ 用户特征提取完成")
print(f"新增列: {[f'{f}_mode' for f in features]}")

✅ 用户特征提取完成
新增列: ['click_environment_mode', 'click_deviceGroup_mode', 'click_os_mode', 'click_country_mode', 'click_region_mode', 'click_referrer_type_mode']


In [10]:
# 查看处理后的数据
print(all_data[['user_id', 'click_article_id'] + [f'{f}_mode' for f in features]].head())

   user_id  click_article_id  click_environment_mode  click_deviceGroup_mode  \
0        0             30760                       4                       1   
1        0            157507                       4                       1   
2        1            289197                       4                       1   
3        1             63746                       4                       1   
4        2             36162                       4                       3   

   click_os_mode  click_country_mode  click_region_mode  \
0             17                   1                 25   
1             17                   1                 25   
2             17                   1                 25   
3             17                   1                 25   
4             20                   1                 25   

   click_referrer_type_mode  
0                         2  
1                         2  
2                         6  
3                         6  
4             

## 步骤3：特征编码（LabelEncoding）

In [28]:
# 定义特征列
user_cols = ["user_id", 'click_environment_mode', 'click_deviceGroup_mode',
             'click_os_mode', 'click_country_mode', 'click_region_mode', 'click_referrer_type_mode']
item_cols = ['click_article_id', "category_id", "words_count", "created_at_ts"]
sparse_features = ['user_id', 'click_article_id', 'category_id', 'click_environment_mode',
                   'click_deviceGroup_mode', 'click_os_mode', 'click_country_mode',
                   'click_region_mode', 'click_referrer_type_mode']

print(f"稀疏特征: {sparse_features}")

稀疏特征: ['user_id', 'click_article_id', 'category_id', 'click_environment_mode', 'click_deviceGroup_mode', 'click_os_mode', 'click_country_mode', 'click_region_mode', 'click_referrer_type_mode']


In [29]:
# LabelEncoding
feature_max_idx = {}
user_map = {}
item_map = {}

print("编码前样本:")
print(all_data[sparse_features].head())

for feature in sparse_features:
    lbe = LabelEncoder()
    all_data[feature] = lbe.fit_transform(all_data[feature]) + 1  # +1避免0
    feature_max_idx[feature] = all_data[feature].max() + 1
    
    if feature == 'user_id':
        user_map = {encode_id + 1: raw_id for encode_id, raw_id in enumerate(lbe.classes_)}
    if feature == 'click_article_id':
        item_map = {encode_id + 1: raw_id for encode_id, raw_id in enumerate(lbe.classes_)}

print("\n编码后样本:")
print(all_data[sparse_features].head())
print(f"\n特征最大索引: {feature_max_idx}")

编码前样本:
   user_id  click_article_id  category_id  click_environment_mode  \
0        1              2343           17                       3   
1        1             16105          180                       3   
2        2             29121          268                       3   
3        2              6678           82                       3   
4        3              3222           30                       3   

   click_deviceGroup_mode  click_os_mode  click_country_mode  \
0                       1              6                   1   
1                       1              6                   1   
2                       1              6                   1   
3                       1              6                   1   
4                       2              8                   1   

   click_region_mode  click_referrer_type_mode  
0                 25                         2  
1                 25                         2  
2                 25                         6

## 步骤4：构建历史序列特征

In [30]:
# 构建用户历史点击序列
seq_max_len = 30  # 最大序列长度

user_hist = all_data.groupby('user_id').agg({
    'click_article_id': lambda x: list(x)[:-1][-seq_max_len:],  # 历史点击（去掉最后一个）
    'category_id': lambda x: list(x)[:-1][-seq_max_len:],
}).reset_index()

user_hist.columns = ['user_id', 'hist_click_article_id', 'hist_category_id']
user_hist['hist_len'] = user_hist['hist_click_article_id'].apply(len)

print(f"用户历史序列数: {len(user_hist)}")
print(user_hist.head())

用户历史序列数: 250000
   user_id hist_click_article_id hist_category_id  hist_len
0        1                [2343]             [17]         1
1        2               [29121]            [268]         1
2        3                [3222]             [30]         1
3        4                [5196]             [62]         1
4        5                [4359]             [45]         1


In [35]:
# 统计序列长度分布
print(user_hist['hist_len'].describe())
user_hist.head(10)

count    250000.000000
mean          5.177168
std           6.406450
min           0.000000
25%           1.000000
50%           3.000000
75%           6.000000
max          30.000000
Name: hist_len, dtype: float64


,user_id,hist_click_article_id,hist_category_id,hist_len
0,1,[2343],[17],1
1,2,[29121],[268],1
2,3,[3222],[30],1
3,4,[5196],[62],1
4,5,[4359],[45],1
5,6,[21611],[217],1
6,7,[6434],[78],1
7,8,[5196],[62],1
8,9,[7708],[85],1
9,10,"[7708, 21611]","[85, 217]",2


## 步骤5：准备训练数据

In [36]:
# 划分训练样本（每个用户最后一次点击作为正样本）
train_samples = all_data.groupby('user_id').tail(1).reset_index(drop=True)
train_samples['label'] = 1  # 正样本标签

print(f"训练样本数: {len(train_samples)}")
print(train_samples[['user_id', 'click_article_id', 'label']].head())
train_samples.head(10)

训练样本数: 250000
   user_id  click_article_id  label
0        1             16105      1
1        2              6678      1
2        3             17493      1
3        4              3222      1
4        5              3789      1


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type,category_id,created_at_ts,words_count,click_environment_mode,click_deviceGroup_mode,click_os_mode,click_country_mode,click_region_mode,click_referrer_type_mode,label
0,1,16105,1508211702520,4,1,17,1,25,2,180,0.964860,0.055306,3,1,6,1,25,2,1
1,2,6678,1508211346889,4,1,17,1,25,6,82,0.964599,0.024215,3,1,6,1,25,6,1
2,3,17493,1508211468695,4,3,20,1,25,2,189,0.963274,0.032138,3,2,8,1,25,2,1
3,4,3222,1508211389672,4,3,2,1,25,2,30,0.964695,0.030643,3,2,1,1,25,2,1
4,5,3789,1508211655466,4,1,12,1,16,1,44,0.964627,0.026158,3,1,4,1,16,1,1
5,6,23679,1508211273884,4,4,2,1,25,2,236,0.964707,0.031689,3,3,1,1,25,2,1
6,7,860,1508212170776,4,3,2,1,2,1,4,0.964674,0.031689,3,2,1,1,2,1,1
7,8,21611,1508211343878,4,1,17,1,25,2,217,0.964724,0.035874,3,1,6,1,25,2,1
8,9,5196,1508211202391,4,1,17,1,21,2,62,0.964709,0.028102,3,1,6,1,21,2,1
9,10,21613,1508211275547,4,3,2,1,25,2,217,0.964345,0.022870,3,2,1,1,25,2,1


In [37]:
# 合并历史序列
train_data_final = train_samples.merge(user_hist, on='user_id', how='left')

print(f"最终训练数据: {train_data_final.shape}")
print(train_data_final[['user_id', 'click_article_id', 'hist_len', 'label']].head())
train_data_final.head(10)

最终训练数据: (250000, 22)
   user_id  click_article_id  hist_len  label
0        1             16105         1      1
1        2              6678         1      1
2        3             17493         1      1
3        4              3222         1      1
4        5              3789         1      1


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type,category_id,...,click_environment_mode,click_deviceGroup_mode,click_os_mode,click_country_mode,click_region_mode,click_referrer_type_mode,label,hist_click_article_id,hist_category_id,hist_len
0,1,16105,1508211702520,4,1,17,1,25,2,180,...,3,1,6,1,25,2,1,[2343],[17],1
1,2,6678,1508211346889,4,1,17,1,25,6,82,...,3,1,6,1,25,6,1,[29121],[268],1
2,3,17493,1508211468695,4,3,20,1,25,2,189,...,3,2,8,1,25,2,1,[3222],[30],1
3,4,3222,1508211389672,4,3,2,1,25,2,30,...,3,2,1,1,25,2,1,[5196],[62],1
4,5,3789,1508211655466,4,1,12,1,16,1,44,...,3,1,4,1,16,1,1,[4359],[45],1
5,6,23679,1508211273884,4,4,2,1,25,2,236,...,3,3,1,1,25,2,1,[21611],[217],1
6,7,860,1508212170776,4,3,2,1,2,1,4,...,3,2,1,1,2,1,1,[6434],[78],1
7,8,21611,1508211343878,4,1,17,1,25,2,217,...,3,1,6,1,25,2,1,[5196],[62],1
8,9,5196,1508211202391,4,1,17,1,21,2,62,...,3,1,6,1,21,2,1,[7708],[85],1
9,10,21613,1508211275547,4,3,2,1,25,2,217,...,3,2,1,1,25,2,1,"[7708, 21611]","[85, 217]",2


## 步骤6：构建DSSM模型（需要安装deepmatch）

In [38]:
# 检查是否可以构建模型
try:
    # 定义用户特征
    user_features = [SparseFeat('user_id', feature_max_idx['user_id'], 32)]
    user_features += [SparseFeat(feat, feature_max_idx[feat], 16) for feat in user_cols[1:]]
    
    # 序列特征
    user_features += [
        VarLenSparseFeat(
            SparseFeat("hist_click_article_id", feature_max_idx["click_article_id"], 32, 
                      embedding_name="click_article_id"),
            seq_max_len, 'sum'
        ),
        VarLenSparseFeat(
            SparseFeat("hist_category_id", feature_max_idx["category_id"], 32,
                      embedding_name="category_id"),
            seq_max_len, 'sum'
        ),
    ]
    
    # 物品特征
    item_sparse = ['click_article_id', 'category_id']
    item_dense = ['words_count', 'created_at_ts']
    item_features = [SparseFeat(feat, feature_max_idx[feat], 32) for feat in item_sparse]
    item_features += [DenseFeat(feat) for feat in item_dense]
    
    print("✅ 特征定义成功！")
    print(f"用户特征数: {len(user_features)}")
    print(f"物品特征数: {len(item_features)}")
    
except Exception as e:
    print(f"⚠️ 特征定义失败: {e}")

✅ 特征定义成功！
用户特征数: 9
物品特征数: 4


In [39]:
# 构建模型（如果依赖已安装）
try:
    model = DSSM(
        user_features, item_features,
        user_dnn_hidden_units=(64, 32),
        item_dnn_hidden_units=(64, 32),
        loss_type='softmax'
    )
    model.compile(optimizer='adam', loss=sampledsoftmaxloss)
    print("✅ DSSM模型构建成功！")
    model.summary()
except Exception as e:
    print(f"⚠️ 模型构建失败: {e}")
    print("请确保已安装: pip install deepmatch tensorflow")

⚠️ 模型构建失败: 'NoneType' object has no attribute '_asdict'
请确保已安装: pip install deepmatch tensorflow


## 数据统计

In [19]:
print("=" * 50)
print("DSSM 数据准备统计")
print("=" * 50)
print(f"总用户数: {all_data['user_id'].nunique()}")
print(f"总物品数: {all_data['click_article_id'].nunique()}")
print(f"训练样本数: {len(train_samples)}")
print(f"序列最大长度: {seq_max_len}")
print(f"用户特征: {len(user_cols)}个")
print(f"物品特征: {len(item_cols)}个")
print("=" * 50)

DSSM 数据准备统计
总用户数: 250000
总物品数: 35380
训练样本数: 250000
序列最大长度: 30
用户特征: 7个
物品特征: 4个


---

## 下一步

1. **模型训练** - 使用 model.fit() 训练DSSM
2. **生成Embedding** - 提取用户和物品的向量
3. **Faiss召回** - 用向量相似度找候选物品

（需要安装 deepmatch 和 tensorflow 才能继续）